# NB-04-01 — Propositional Logic Workbench

**Module:** M04 · **Time:** 60–90 minutes · **Prerequisites:** L04-02, L04-03

You will build truth tables, check the equivalence laws, test entailment by model
enumeration, convert to clause form and run a resolution refutation.

Run each cell in order. Where a cell prints a result, predict it first — the notebook is
for checking your reasoning, not replacing it.

## 1. Setup

In [1]:
import sys
from pathlib import Path

for parent in [Path.cwd(), *Path.cwd().parents]:
    candidate = parent / "modules" / "M04-knowledge-representation-and-logical-reasoning" / "code"
    if candidate.is_dir():
        sys.path.insert(0, str(candidate))
        break
print("code directory:", candidate)

code directory: C:\Users\Nadeem\Documents\Artificial-Intelligence-Course\modules\M04-knowledge-representation-and-logical-reasoning\code


In [2]:
from prop_logic import (
    Symbol, Not, And, Or, Implies, Iff, iff,
    truth_table, is_valid, is_satisfiable, entails,
    counterexample, to_cnf, clauses, clause_str, resolution_refutation,
)

## 2. Truth tables

`truth_table` returns a header row and one row per assignment. The last column is the
formula's value.

In [3]:
rain, wet, indoors = Symbol("rain"), Symbol("wet"), Symbol("indoors")
header, rows = truth_table(rain >> wet)
print(" | ".join(header))
for row in rows:
    print(" | ".join("T" if v else "F" for v in row))

rain | wet | (rain → wet)
F | F | T
F | T | T
T | F | F
T | T | T


**Check:** exactly one row is `F`. Which assignment is it, and why is that the only
world the implication forbids?

### 2.1 Your turn

In [4]:
# Build the table for (rain -> wet) and (wet -> indoors).
# Classify the formula: valid, satisfiable or unsatisfiable?
formula = ""
print(is_satisfiable(formula) if formula else "choose a formula first")

choose a formula first


## 3. The equivalence laws

Each law is checked by asking whether the biconditional is valid.

In [5]:
p, q, r = Symbol("p"), Symbol("q"), Symbol("r")
laws = {
    "De Morgan (and)": iff(~(p & q), ~p | ~q),
    "De Morgan (or)": iff(~(p | q), ~p & ~q),
    "contraposition": iff(p >> q, ~q >> ~p),
    "implication elimination": iff(p >> q, ~p | q),
    "absorption": iff(p | (p & q), p),
    "exportation": iff((p & q) >> r, p >> (q >> r)),
}
for name, formula in laws.items():
    print(f"{name:24s} {is_valid(formula)}")

De Morgan (and)          True
De Morgan (or)           True
contraposition           True
implication elimination  True
absorption               True
exportation              True


Now the two traps: the converse of an implication, and resolution treated as an
equivalence rather than an inference.

In [6]:
print("converse is a law:      ", is_valid(iff(p >> q, q >> p)))
print("resolution equivalence:  ", is_valid(iff((p | q) & (~p | r), q | r)))
print("resolution inference:    ", is_valid(((p | q) & (~p | r)) >> (q | r)))

converse is a law:       False
resolution equivalence:   False
resolution inference:     True


The second line is `False` and the third is `True`. Explain the difference in one
sentence before continuing.

## 4. Entailment by model enumeration

In [7]:
kb = [rain >> wet, wet >> ~indoors, rain]
print("KB entails not indoors?", entails(kb, Not(indoors)))
print("KB entails wet?        ", entails(kb, wet))
print("KB entails indoors?    ", entails(kb, indoors))
print("countermodel for indoors:", counterexample(kb, indoors))

KB entails not indoors? True
KB entails wet?         True
KB entails indoors?     False
countermodel for indoors: {'indoors': False, 'rain': True, 'wet': True}


A countermodel is the most useful output a logic tool can give you: it is a world
where your premises hold and your conclusion fails.

## 5. Clause form

In [8]:
formula = (rain >> wet) >> indoors
print("cnf:", to_cnf(formula))
for clause in clauses(formula):
    print("  clause:", clause_str(clause))

cnf: ((rain ∨ indoors) ∧ (¬wet ∨ indoors))
  clause: indoors ∨ rain
  clause: indoors ∨ ¬wet


## 6. Resolution refutation

In [9]:
proved, trace = resolution_refutation(kb, Not(indoors))
print("proved not indoors:", proved)
for line in trace:
    print("  ", line)

proved not indoors: True
   start: 4 clauses
   resolve ¬rain ∨ wet with ¬indoors ∨ ¬wet -> ¬indoors ∨ ¬rain
   resolve ¬rain ∨ wet with rain -> wet
   resolve ¬indoors ∨ ¬wet with indoors -> ¬wet
   resolve ¬rain ∨ wet with ¬wet -> ¬rain
   resolve ¬indoors ∨ ¬wet with wet -> ¬indoors
   resolve wet with ¬wet -> □


Run it again with `indoors` (the un-negated query). The procedure fails, which is
correct: the KB entails the negation of `indoors`, not `indoors` itself.

In [10]:
proved, trace = resolution_refutation(kb, indoors)
print("proved indoors:", proved)
print("last line:", trace[-1])

proved indoors: False
last line: no new clauses: no refutation exists


## 7. Exercise — an inconsistent KB

Add a rule that contradicts the KB and check whether resolution reports a refutation for
an unrelated query.

In [11]:
inconsistent = kb + [indoors]
print("KB + indoors entails not indoors:", entails(inconsistent, Not(indoors)))
print("KB + indoors entails wet:        ", entails(inconsistent, wet))
print("satisfiable:", is_satisfiable(inconsistent[0] & inconsistent[1] & inconsistent[2] & inconsistent[3]))

KB + indoors entails not indoors: True
KB + indoors entails wet:         True
satisfiable: False


An inconsistent KB entails *everything* — the principle of explosion. Any query you
ask returns `True`, so entailment alone cannot tell you the KB is broken; you have to test
satisfiability separately.

## 8. Reflect

1. Which cell surprised you, and what did you predict instead?
2. Why does `resolution_refutation(kb, indoors)` fail while `resolution_refutation(kb, Not(indoors))` succeeds?
3. What does the countermodel in §4 tell you about `indoors` that the boolean answer does not?

Next: **NB-04-02** for first-order inference, then **LAB-04-01**.